In [18]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

torch: 2.10.0
transformers: 5.1.0


In [24]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from pathlib import Path

# --- Robust project root detection (works manual + automation) ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "frontend":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ART_DIR  = PROJECT_ROOT / "artifacts"

INPUT_PATH = DATA_DIR / "hn_scored_latest.csv"
INDEX_PATH = ART_DIR / "hn_rag.index"
META_PATH  = ART_DIR / "hn_rag_meta.pkl"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_COL = "clean_text"
THEME_COL = "market_theme"

print("Reading scored:", INPUT_PATH)
print("Writing index :", INDEX_PATH)
print("Writing meta  :", META_PATH)

# ---------- Load ----------
df = pd.read_csv(INPUT_PATH)
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
df = df[df[TEXT_COL].str.len() >= 40].copy()

# ---------- Build docs ----------
docs = []
meta = []

for _, r in df.iterrows():
    text = r.get(TEXT_COL, "")
    title = str(r.get("story_title", "") or "")
    theme = str(r.get(THEME_COL, "") or "")
    url = str(r.get("url", "") or "")
    created_at = str(r.get("created_at", "") or "")
    keyword = str(r.get("keyword", "") or "")
    objectID = str(r.get("objectID", "") or "")

    # A little formatting helps retrieval quality
    doc = f"Title: {title}\nTheme: {theme}\nKeyword: {keyword}\nDate: {created_at}\nText: {text}\nURL: {url}"

    docs.append(doc)
    meta.append({
        "story_title": title,
        "market_theme": theme,
        "keyword": keyword,
        "created_at": created_at,
        "url": url,
        "objectID": objectID
    })

print("Docs:", len(docs))

# ---------- Embed ----------
embedder = SentenceTransformer(MODEL_NAME)
emb = embedder.encode(docs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype("float32")

# ---------- FAISS index ----------
d = emb.shape[1]
index = faiss.IndexFlatIP(d)  # cosine similarity if vectors are normalized
index.add(emb)

faiss.write_index(index, str(INDEX_PATH))
with open(META_PATH, "wb") as f:
    pickle.dump({"meta": meta, "docs": docs}, f)

print("Saved:", INDEX_PATH, META_PATH)

Reading scored: /Users/adarshthakur/Downloads/MIANALYZER/data/hn_scored_latest.csv
Writing index : /Users/adarshthakur/Downloads/MIANALYZER/artifacts/hn_rag.index
Writing meta  : /Users/adarshthakur/Downloads/MIANALYZER/artifacts/hn_rag_meta.pkl
Docs: 874


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1471.13it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28/28 [00:02<00:00, 11.63it/s]

Saved: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/hn_rag.index /Users/adarshthakur/Downloads/MIANALYZER/artifacts/hn_rag_meta.pkl


In [25]:
print("INDEX_PATH:", INDEX_PATH)
print("META_PATH :", META_PATH)
print("FAISS ntotal:", index.ntotal)
print("Docs:", len(docs), "Meta:", len(meta))

INDEX_PATH: /Users/adarshthakur/Downloads/MIANALYZER/artifacts/hn_rag.index
META_PATH : /Users/adarshthakur/Downloads/MIANALYZER/artifacts/hn_rag_meta.pkl
FAISS ntotal: 874
Docs: 874 Meta: 874


In [ ]:
import faiss
import pickle
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from datetime import datetime, timezone, timedelta

INDEX_PATH = ART_DIR / "hn_rag.index"
META_PATH  = ART_DIR / "hn_rag_meta.pkl"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 8

# --- Load index ---
index = faiss.read_index(str(INDEX_PATH))

with open(META_PATH, "rb") as f:
    store = pickle.load(f)

docs = store["docs"]
meta = store["meta"]

embedder = SentenceTransformer(MODEL_NAME)

# ---- Filters ----
PAIN_CUES = [
    "pain", "pain point", "problem", "issue", "hard", "difficult", "friction",
    "annoy", "struggle", "fails", "confusing", "complain", "pitfall",
    "refund", "refunds", "chargeback", "chargebacks", "compliance",
    "access control", "entitlement", "entitlements", "permissions", "oauth"
]

def has_pain_language(text: str) -> bool:
    t = str(text).lower()
    return any(w in t for w in PAIN_CUES)

def is_within_days(created_at, days=7) -> bool:
    """
    Returns True if created_at is within the last `days` days.
    Works with ISO timestamps and most parseable formats.
    """
    dt = pd.to_datetime(created_at, utc=True, errors="coerce")
    if pd.isna(dt):
        return False
    return dt >= (datetime.now(timezone.utc) - timedelta(days=days))

def search(
    query: str,
    top_k: int = TOP_K,
    theme_filter: str | None = None,
    days: int | None = 7,
    require_pain: bool = True
):
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")

    # Over-retrieve so filtering doesn't starve results
    overfetch = max(top_k * 50, 200)
    scores, idx = index.search(q, overfetch)

    results = []
    for sc, i in zip(scores[0], idx[0]):
        if i == -1:
            continue

        m = meta[i]
        doc = docs[i]

        # ✅ This-week filter
        if days is not None and not is_within_days(m.get("created_at", ""), days=days):
            continue

        # ✅ Optional theme filter
        if theme_filter and m.get("market_theme") != theme_filter:
            continue

        # ✅ Pain-aware filter
        if require_pain and not has_pain_language(doc):
            continue

        results.append((float(sc), m, doc))
        if len(results) >= top_k:
            break

    return results

if __name__ == "__main__":
    print("RAG ready. Type a question (or 'exit' to quit).\n")

    while True:
        q = input("Ask: ").strip()
        if q.lower() in {"exit", "quit", "q"}:
            print("Goodbye 👋")
            break

        results = search(
            q,
            top_k=8,
            days=7,
            require_pain=True
        )

        if not results:
            print("\nNo relevant results found for this week.\n")
            continue

        for rank, (sc, m, doc) in enumerate(results, start=1):
            print("\n" + "="*80)
            print(f"{rank}) score={sc:.3f} | theme={m.get('market_theme')} | keyword={m.get('keyword')}")
            print(f"   title: {m.get('story_title')}")
            print(f"   url:   {m.get('url')}")
            print("-"*80)
            text_part = doc.split("\nText:", 1)[-1].strip()
            print(text_part[:700], "..." if len(text_part) > 700 else "")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2101.44it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG ready. Type a question (or 'exit' to quit).



Ask:  wow



1) score=0.196 | theme=Platform Perception & Social Sentiment | keyword=film
   title: New accounts on HN more likely to use em-dashes
   url:   https://news.ycombinator.com/item?id=47160803
--------------------------------------------------------------------------------
New accounts on HN more likely to use em-dashes. > The filter used to be effort. You had to care enough to spend weeks on something, which meant you probably understood the problem deeply. Now that filter is gone and we get a flood of "I prompted this in 20 minutes" posts where the author can't answer a single follow-up about their own code. The interesting Show HNs still exist, they're just buried under noise. The irony of a bot comment
URL: https://news.ycombinator.com/item?id=47160803 

2) score=0.189 | theme=Early Product Feedback & Tool Discovery | keyword=creator
   title: Loops is a federated, open-source TikTok
   url:   https://news.ycombinator.com/item?id=47116578
--------------------------------------------

In [11]:
print("Index exists:", INDEX_PATH.exists())
print("Meta exists:", META_PATH.exists())

Index exists: True
Meta exists: True
